## Step up cryptographic environment

In [1]:
%pip install cryptography

In [2]:
import os
import json
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag


In [3]:
import cryptography

print("Cryptography version:", cryptography.__version__)
print("AES-GCM ready")

Cryptography version: 50.0.2
AES-GCM ready


## Defining the encryption protocol

In [4]:
class HealthcareProtocol:

    def __init__(self, flawed=False):

        self.key = AESGCM.generate_key(bit_length=256)
        self.aesgcm = AESGCM(self.key)

        self.counter = 0
        self.flawed = flawed

    def encrypt(self, plaintext, metadata):

        if self.counter >= 2**96:
            raise ValueError("Nonce space exhausted")

        # Reusing nonce in the flawed implementation
        nonce_number = 0 if self.flawed else self.counter

        nonce = nonce_number.to_bytes(12, "big")

        self.counter += 1

        aad = json.dumps(
            metadata, sort_keys=True
        ).encode("utf-8")

        encrypted = self.aesgcm.encrypt(
            nonce,
            plaintext.encode("utf-8"),
            aad
        )

        return {
            "nonce": nonce,
            "ciphertext": encrypted[:-16],
            "tag": encrypted[-16:],
            "aad": aad
        }

    def decrypt(self, packet):

        plaintext = self.aesgcm.decrypt(
            packet["nonce"],
            packet["ciphertext"] + packet["tag"],
            packet["aad"]
        )

        return plaintext.decode("utf-8")

### Patient metadata

In [5]:
metadata = {
    "patient_id": "P1001",
    "sender": "Hospital_A",
    "recipient": "GP_B",
    "key_version": 1
}

### Testing secure encryption

In [6]:
secure = HealthcareProtocol(flawed=False)

patient_record = "Patient requires follow-up appointment"

packet = secure.encrypt(patient_record, metadata)

print("Nonce:", packet["nonce"].hex())
print("Ciphertext:", packet["ciphertext"].hex())
print("Authentication Tag:", packet["tag"].hex())

decrypted = secure.decrypt(packet)

print("Decrypted Record:", decrypted)
print("Successful:", decrypted == patient_record)

Nonce: 000000000000000000000000
Ciphertext: b7151acc174480e683faee1a100f14a6fb132df3b19c4951738165d55b82eb63d734c18101cf
Authentication Tag: 8f21d8f19a8cfcb7b985e1735a3a6087
Decrypted Record: Patient requires follow-up appointment
Successful: True


## Security Testing
### Creating test data

In [7]:
test_record = "Patient requires urgent medical review"

test_packet = secure.encrypt(test_record, metadata)

print("Test packet generated")

Test packet generated


### Testing successful decryption

In [8]:
result = secure.decrypt(test_packet)

print("Original:", test_record)
print("Decrypted:", result)
print("Test passed:", result == test_record)

Original: Patient requires urgent medical review
Decrypted: Patient requires urgent medical review
Test passed: True


### Testing modified ciphertext

In [9]:
tampered_packet = test_packet.copy()

tampered_ciphertext = bytearray(tampered_packet["ciphertext"])
tampered_ciphertext[0] ^= 1

tampered_packet["ciphertext"] = bytes(tampered_ciphertext)

try:
    secure.decrypt(tampered_packet)
    print("Test failed: Modified ciphertext accepted")

except InvalidTag:
    print("Test passed: Modified ciphertext rejected")

Test passed: Modified ciphertext rejected


### Testing modified metadata

In [10]:
modified_packet = test_packet.copy()

modified_packet["aad"] = json.dumps({
    "patient_id": "P2002",
    "sender": "Hospital_A",
    "recipient": "GP_B",
    "key_version": 1
}, sort_keys=True).encode("utf-8")

try:
    secure.decrypt(modified_packet)
    print("Test failed: Modified metadata accepted")

except InvalidTag:
    print("Test passed: Modified metadata rejected")

Test passed: Modified metadata rejected


### Testing incorrect key

In [11]:
wrong_key = AESGCM.generate_key(bit_length=256)
wrong_aesgcm = AESGCM(wrong_key)

try:
    wrong_aesgcm.decrypt(
        test_packet["nonce"],
        test_packet["ciphertext"] + test_packet["tag"],
        test_packet["aad"]
    )

    print("Test failed: Incorrect key accepted")

except InvalidTag:
    print("Test passed: Incorrect key rejected")

Test passed: Incorrect key rejected


### Testing results

In [12]:
print("SECURITY TEST SUMMARY")
print("---------------------")
print("Successful decryption: Expected to pass")
print("Modified ciphertext: Expected rejection")
print("Modified metadata: Expected rejection")
print("Incorrect key: Expected rejection")

SECURITY TEST SUMMARY
---------------------
Successful decryption: Expected to pass
Modified ciphertext: Expected rejection
Modified metadata: Expected rejection
Incorrect key: Expected rejection


Creating the flawed implementation

In [14]:
flawed = HealthcareProtocol(flawed=True)

print("Flawed implementation created")
print("Nonce reuse enabled:", flawed.flawed)

Flawed implementation created
Nonce reuse enabled: True


Creating two patient record

In [15]:
known_plaintext = "Patient record: NORMAL"
secret_plaintext = "Patient record: SECRET"

print("Known plaintext:", known_plaintext)
print("Secret plaintext: [HIDDEN]")

Known plaintext: Patient record: NORMAL
Secret plaintext: [HIDDEN]


Encrypting both records

In [16]:
packet1 = flawed.encrypt(known_plaintext, metadata)
packet2 = flawed.encrypt(secret_plaintext, metadata)

print("First nonce:", packet1["nonce"].hex())
print("Second nonce:", packet2["nonce"].hex())

print("First ciphertext:", packet1["ciphertext"].hex())
print("Second ciphertext:", packet2["ciphertext"].hex())

First nonce: 000000000000000000000000
Second nonce: 000000000000000000000000
First ciphertext: f823644a15958264444e8f1a0103f5d33b8f7514cbfb
Second ciphertext: f823644a15958264444e8f1a0103f5d32685640bcfe3


Implementing the XOR attack

In [18]:
def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))


def recover_plaintext(ciphertext1, ciphertext2, known_text):

    known_bytes = known_text.encode("utf-8")

    recovered = xor_bytes(
        xor_bytes(ciphertext1, ciphertext2),
        known_bytes
    )

    return recovered

Executing the attack

In [20]:
recovered_plaintext = recover_plaintext(
    packet1["ciphertext"],
    packet2["ciphertext"],
    known_plaintext
)

print("Known plaintext:", known_plaintext)
print("Recovered plaintext:", recovered_plaintext.decode("utf-8"))
print("Actual secret plaintext:", secret_plaintext)

print(
    "Attack successful:",
    recovered_plaintext == secret_plaintext.encode("utf-8")
)

Known plaintext: Patient record: NORMAL
Recovered plaintext: Patient record: SECRET
Actual secret plaintext: Patient record: SECRET
Attack successful: True


Running the same attack against both

In [21]:
def run_attack_test(protocol, label):

    first = protocol.encrypt(known_plaintext, metadata)
    second = protocol.encrypt(secret_plaintext, metadata)

    recovered = recover_plaintext(
        first["ciphertext"],
        second["ciphertext"],
        known_plaintext
    )

    success = recovered == secret_plaintext.encode("utf-8")

    print("\n", label)
    print("Nonce 1:", first["nonce"].hex())
    print("Nonce 2:", second["nonce"].hex())
    print("Recovered plaintext:", recovered.decode("utf-8", errors="replace"))
    print("Attack successful:", success)

    return success

Comparative attack results

In [22]:
flawed_result = run_attack_test(
    HealthcareProtocol(flawed=True),
    "FLAWED IMPLEMENTATION"
)

secure_result = run_attack_test(
    HealthcareProtocol(flawed=False),
    "SECURE IMPLEMENTATION"
)

print("\nFINAL COMPARISON")
print("----------------")
print("Flawed attack succeeded:", flawed_result)
print("Secure attack succeeded:", secure_result)

assert flawed_result is True
assert secure_result is False

print("\nBoth comparative tests passed")


 FLAWED IMPLEMENTATION
Nonce 1: 000000000000000000000000
Nonce 2: 000000000000000000000000
Recovered plaintext: Patient record: SECRET
Attack successful: True

 SECURE IMPLEMENTATION
Nonce 1: 000000000000000000000000
Nonce 2: 000000000000000000000001
Recovered plaintext: ��r��8��Ī��~2��D�
Attack successful: False

FINAL COMPARISON
----------------
Flawed attack succeeded: True
Secure attack succeeded: False

Both comparative tests passed


## Experimental results

In [23]:
print("CRYPTOGRAPHIC ATTACK SUMMARY")
print("=" * 40)

print("Protocol: AES-256-GCM")
print("Vulnerability: Nonce reuse")
print("Attack: Known-plaintext XOR attack")

print("\nFlawed implementation:")
print("Nonce reused: Yes")
print("Plaintext recovered: Yes")

print("\nSecure implementation:")
print("Nonce reused: No")
print("Plaintext recovered: No")

print("\nConclusion:")
print("Nonce reuse compromises confidentiality.")

CRYPTOGRAPHIC ATTACK SUMMARY
Protocol: AES-256-GCM
Vulnerability: Nonce reuse
Attack: Known-plaintext XOR attack

Flawed implementation:
Nonce reused: Yes
Plaintext recovered: Yes

Secure implementation:
Nonce reused: No
Plaintext recovered: No

Conclusion:
Nonce reuse compromises confidentiality.
